# Sutra: PACE-ICE prechecks and demo

Use a **Python 3 kernel in an allocated PACE-ICE compute-node Jupyter session**. C++ stays in source files; this notebook drives commands and displays evidence. Start through [Open OnDemand](https://ondemand-ice.pace.gatech.edu). Local Windows execution can inspect the repository but cannot establish ICE/GPU readiness.

**Available now:** repository inspection, system/compiler/CUDA probes, optional pinned llama.cpp CPU/CUDA baseline builds and smoke runs. **Pending implementation:** the Sutra runtime and `scripts/run_demo.sh`. A llama.cpp smoke run is not an end-to-end Sutra demonstration.

Read [IMPLEMENTATION_GUIDE.md](../IMPLEMENTATION_GUIDE.md) for implementation steps and [PROJECT_DELIVERY_PLAN.md](../PROJECT_DELIVERY_PLAN.md) for contracts/acceptance gates. Course references: CUDA starter notebook setup cell 3, CUDA Intro cell 9, VS Code on PACE-ICE guide, and disk-quota notebook in the parent CourseMaterial folder. They specify system-toolchain CUDA builds, Release timing, compute-node execution and quota checks.

Run cells top to bottom. First leave expensive operations disabled, inspect prechecks, then enable the desired stage. Each command has a timeout, saved log and subprocess-group cleanup on interrupt. The notebook does not request new allocations, overwrite source files, download model weights or start an unbounded server.

## 1. Configuration

If auto-discovery fails, set `REPO_OVERRIDE` to the absolute **Linux** repository path. Stage the team's GGUF first. Use a fresh session outside the course `ece4122-6122` conda build environment; a normal Python kernel is sufficient. Inspect current module availability if `cuda` is not the correct name. Do not override Slurm's `CUDA_VISIBLE_DEVICES`.

In [ ]:
from pathlib import Path
import csv, datetime, hashlib, json, math, os, platform, shutil, signal
import socket, statistics, subprocess, time

REPO_OVERRIDE = None  # e.g. Path('/your/approved/path/SUTRA-APT6122Project')
MODEL_PATH = None  # e.g. Path('/your/approved/path/qwen2.5-0.5b-instruct-q4_k_m.gguf')
MODEL_SOURCE = ''  # publisher/repository or team artifact reference
MODEL_REVISION = ''  # record exact model revision when known
CUDA_MODULE = 'cuda'  # may be a versioned module on your ICE session
SYSTEM_CC = '/usr/bin/gcc'
SYSTEM_CXX = '/usr/bin/g++'  # choose a supported module compiler here if required
CUDA_ARCH = None  # set after inspecting your GPU, e.g. '70' for V100; no assumed GPU
BUILD_JOBS = 4  # must fit your allocation; helper clamps to process CPU affinity
RUN_PROBES = True
INIT_DEPENDENCY = False  # enable only if the recorded submodule is not initialized
RUN_BASELINE = False  # enables builds, short text generation and microbenchmarks
RUN_SUTRA = False  # enable only after scripts/run_demo.sh is implemented
DEMO_BACKEND = 'mock'  # later: 'cuda'
SAVED_RESULTS_DIR = None  # later: Path('/.../results/<run_id>')

candidates = [Path(REPO_OVERRIDE).expanduser()] if REPO_OVERRIDE else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p/'common/inference_types.hpp').is_file() and (p/'scripts/ice_env.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Set REPO_OVERRIDE to the repository containing scripts/ice_env.sh.')
if MODEL_PATH is not None:
    MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()
IS_ICE_ALLOCATION = platform.system() == 'Linux' and bool(os.environ.get('SLURM_JOB_ID'))
affinity_count = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
JOBS = max(1, min(BUILD_JOBS, affinity_count))
RUN_ID = 'notebook-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + str(os.getpid())
RUN_DIR = ROOT/'results'/RUN_ID
print('Repository:', ROOT)
print('Host:', socket.gethostname(), '| OS:', platform.system(), '| Slurm job:', os.environ.get('SLURM_JOB_ID', 'NONE'))
print('Build jobs:', JOBS, '| Output directory (created on first command):', RUN_DIR)
print('Allocation guard:', 'PASS: inspect GPU/hostname below too' if IS_ICE_ALLOCATION else 'NOT ON ICE: compute cells will skip')

## 2. Command runner

Every CUDA command reloads its module in its own shell via the shared wrapper; environment changes in an earlier shell cell are not assumed to persist. Build tools use configured system C/C++ compiler paths. If you interrupt a cell, its process group is terminated; do not close a live session in the middle of a run. Long unattended experiments should use Slurm batch jobs instead.

In [ ]:
def run_ice(args, *, cuda=False, timeout=1800, check=True):
    if not IS_ICE_ALLOCATION:
        raise RuntimeError('Launch a remote allocated ICE compute-node session first.')
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env.update(SUTRA_WITH_CUDA='1' if cuda else '0', SUTRA_CUDA_MODULE=CUDA_MODULE,
               SUTRA_CC=SYSTEM_CC, SUTRA_CXX=SYSTEM_CXX)
    command = ['bash', str(ROOT/'scripts/ice_env.sh'), *[str(x) for x in args]]
    log = RUN_DIR/(str(time.time_ns()) + '.log')
    print('Running:', ' '.join(map(str, args)), flush=True)
    print('Log:', log, flush=True)
    with log.open('w', encoding='utf-8') as out:
        proc = subprocess.Popen(command, cwd=ROOT, env=env, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            rc = proc.wait(timeout=timeout)
        except BaseException:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                proc.wait()
            print(log.read_text(encoding='utf-8', errors='replace')[-12000:])
            raise
    output = log.read_text(encoding='utf-8', errors='replace')
    print(output[-12000:])
    print('Exit:', rc)
    if check and rc:
        raise RuntimeError(f'Command failed (exit {rc}); see {log}')
    return output

print('Runner ready; no command launched.')

## 3. Repository status (works locally too)

An empty CMake file is **pending implementation**, not a successful build. The recorded llama.cpp revision remains authoritative; do not automatically upgrade it to upstream master. Initializing the submodule needs network access and a Git checkout.

In [ ]:
for relative in ['CMakeLists.txt', 'common/inference_types.hpp', 'inference/inference_engine.hpp',
                 'worker/worker_main.cpp', 'benchmark/benchmark_client.cpp', 'scripts/run_demo.sh']:
    p = ROOT/relative
    print(f'{relative}: ' + (f'{p.stat().st_size} bytes' if p.is_file() else 'NOT IMPLEMENTED'))
if shutil.which('git'):
    for git_args in [['log', '-1', '--format=%h %s'], ['submodule', 'status']]:
        r = subprocess.run(['git', '-C', str(ROOT), *git_args], capture_output=True, text=True, timeout=15)
        print((r.stdout + r.stderr).strip())
else:
    print('Git not found: install/use the ICE Git tool before dependency initialization.')

## 4. Allocation, quota, compiler and driver checks

Inspect hostname, allocation, compiler identity, module output, quota bytes/files and actual device model. `nvidia-smi` may enumerate more devices than your job is permitted to use; the CUDA-runtime probe below checks devices available to this process. Tool absence and quota warnings are diagnostics; the actual compile/runtime probes are the readiness gates.

In [ ]:
if IS_ICE_ALLOCATION:
    run_ice(['bash', '-c', r"""
set -e
hostname
printf 'Slurm job: %s; visible GPU selection: %s\n' "${SLURM_JOB_ID:-NONE}" "${CUDA_VISIBLE_DEVICES:-scheduler default}"
printf 'C compiler: %s; C++ compiler: %s\n' "$CC" "$CXX"
"$CXX" --version
cmake --version
nvcc --version
if type module >/dev/null 2>&1; then module list 2>&1; fi
nvidia-smi --query-gpu=name,uuid,memory.total,driver_version --format=csv
if command -v pace-quota >/dev/null 2>&1; then pace-quota; elif command -v quota >/dev/null 2>&1; then quota -s; else printf 'Quota tool unavailable: check via course instructions.\n'; fi
df -h .
"""], cuda=True, timeout=60)
else:
    print('SKIP: allocation/toolchain/driver check requires the remote ICE session.')

## 5. Compile and run a C++17/thread probe

This is an independent environment probe, not a Sutra test. It uses the same wrapper/compiler as later builds and writes only generated files beneath this run's ignored results directory.

In [ ]:
CPP_PROBE_OK = False
if IS_ICE_ALLOCATION and RUN_PROBES:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    src = RUN_DIR/'cpp_probe.cpp'
    exe = RUN_DIR/'cpp_probe'
    src.write_text('#include <atomic>\n#include <iostream>\n#include <thread>\nint main(){ std::atomic<int> x{0}; std::thread t([&]{x=42;}); t.join(); if(x!=42)return 1; std::cout<<"PASS C++17 + thread\\n"; }\n', encoding='utf-8')
    run_ice([SYSTEM_CXX, '-std=c++17', '-O2', '-pthread', str(src), '-o', str(exe)], timeout=60)
    run_ice([exe], timeout=15)
    CPP_PROBE_OK = True
else:
    print('SKIP: use ICE allocation and RUN_PROBES=True.')

## 6. Compile and run a real CUDA kernel

Checks compilation, allocation, launch, synchronization and device-to-host copy. Device count/name come from the CUDA runtime under the job's visibility settings. A passing driver check alone is insufficient. Set `CUDA_ARCH` if the toolkit/default architecture requires it, using the device's compute capability, not a guessed value.

In [ ]:
CUDA_PROBE_OK = False
if IS_ICE_ALLOCATION and RUN_PROBES:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    src = RUN_DIR/'cuda_probe.cu'
    exe = RUN_DIR/'cuda_probe'
    src.write_text(r"""
#include <cuda_runtime.h>
#include <iostream>
#define CHECK(x) do { cudaError_t e=(x); if(e!=cudaSuccess){std::cerr<<cudaGetErrorString(e)<<"\n";return 1;} }while(0)
__global__ void probe(int* p){*p=42;}
int main(){
 int count=0; CHECK(cudaGetDeviceCount(&count));
 if(count<1){std::cerr<<"No allocated CUDA device\n";return 1;}
 CHECK(cudaSetDevice(0)); cudaDeviceProp prop{}; CHECK(cudaGetDeviceProperties(&prop,0));
 std::cout<<"Runtime-visible GPUs: "<<count<<"; selected "<<prop.name<<"; compute "<<prop.major<<'.'<<prop.minor<<"\n";
 int* device=nullptr; CHECK(cudaMalloc(reinterpret_cast<void**>(&device),sizeof(int)));
 probe<<<1,1>>>(device); CHECK(cudaGetLastError()); CHECK(cudaDeviceSynchronize());
 int host=0; CHECK(cudaMemcpy(&host,device,sizeof(int),cudaMemcpyDeviceToHost)); CHECK(cudaFree(device));
 if(host!=42)return 1; std::cout<<"PASS CUDA kernel + copy\n";
}
""", encoding='utf-8')
    cmd = ['nvcc', '-std=c++17', '-O2', '-ccbin', SYSTEM_CXX]
    if CUDA_ARCH:
        cmd += ['-arch=sm_' + str(CUDA_ARCH)]
    run_ice(cmd + [str(src), '-o', str(exe)], cuda=True, timeout=120)
    run_ice([exe], cuda=True, timeout=30)
    CUDA_PROBE_OK = True
else:
    print('SKIP: use ICE allocation and RUN_PROBES=True.')

## 7. Initialize dependency and inspect staged model

Set `INIT_DEPENDENCY=True` only when needed. This checks out the superproject's recorded dependency; it does not request the latest upstream commit. Model weights are supplied separately by the team; check provenance and storage quota before transfer. The model SHA-256 is computed by streaming, not loading the entire file into memory.

In [ ]:
if IS_ICE_ALLOCATION and INIT_DEPENDENCY:
    run_ice(['git', 'submodule', 'update', '--init', '--recursive'], timeout=1200)
else:
    print('Dependency initialization skipped (enable INIT_DEPENDENCY if needed).')
LLAMA = ROOT/'external/llama.cpp'
print('Dependency initialized:', (LLAMA/'CMakeLists.txt').is_file())
MODEL_SHA256 = None
if MODEL_PATH is not None and MODEL_PATH.is_file():
    with MODEL_PATH.open('rb') as stream:
        if stream.read(4) != b'GGUF':
            raise RuntimeError('Model is not a GGUF file (magic header check).')
        stream.seek(0)
        digest = hashlib.sha256()
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            digest.update(chunk)
    MODEL_SHA256 = digest.hexdigest()
    print('Model bytes:', MODEL_PATH.stat().st_size, '| SHA-256:', MODEL_SHA256)
    print('Source:', MODEL_SOURCE or 'RECORD SOURCE', '| revision:', MODEL_REVISION or 'RECORD REVISION')
else:
    print('Model not configured/found: set MODEL_PATH to a staged absolute Linux path.')

## 8. Optional llama.cpp baseline builds

Set `RUN_BASELINE=True` after probes pass. Builds use separate Release directories and a bounded CPU count. They build upstream `llama-cli` and `llama-bench`, independently of the currently empty Sutra CMake file. Check the pinned revision's build documentation if requirements differ from current upstream. First compile can take several minutes; full logs are saved.

In [ ]:
BASELINE_BUILT = False
if IS_ICE_ALLOCATION and RUN_BASELINE:
    if not (CPP_PROBE_OK and CUDA_PROBE_OK):
        raise RuntimeError('Run and pass both compiler/runtime probes first.')
    if not (LLAMA/'CMakeLists.txt').is_file():
        raise RuntimeError('Initialize the pinned llama.cpp submodule first.')
    if MODEL_SHA256 is None:
        raise RuntimeError('Set a valid staged MODEL_PATH first.')
    for backend in ['cpu', 'cuda']:
        build = ROOT/('build-llama-' + backend)
        configure = ['cmake', '-S', str(LLAMA), '-B', str(build), '-DCMAKE_BUILD_TYPE=Release',
                     '-DCMAKE_C_COMPILER=' + SYSTEM_CC, '-DCMAKE_CXX_COMPILER=' + SYSTEM_CXX,
                     '-DGGML_CUDA=' + ('ON' if backend == 'cuda' else 'OFF')]
        if backend == 'cuda':
            configure += ['-DCMAKE_CUDA_HOST_COMPILER=' + SYSTEM_CXX]
            if CUDA_ARCH:
                configure += ['-DCMAKE_CUDA_ARCHITECTURES=' + str(CUDA_ARCH)]
        run_ice(configure, cuda=backend == 'cuda', timeout=300)
        run_ice(['cmake', '--build', str(build), '--config', 'Release', '--parallel', str(JOBS),
                 '--target', 'llama-cli', 'llama-bench'], cuda=backend == 'cuda', timeout=1800)
    BASELINE_BUILT = True
else:
    print('SKIP baseline builds: RUN_BASELINE=False or not in an ICE allocation.')

## 9. CPU and CUDA text-generation smoke runs

Both use the same prompt, context size and greedy decoding. The GPU run requests full layer offload; **inspect startup logs for actual offloaded layers and device**. Model loading is included in these short invocations: do not use their wall time as steady-state Sutra throughput. CLI help is checked for non-conversation mode so the cell cannot silently become an interactive chat session.

In [ ]:
if BASELINE_BUILT:
    for backend in ['cpu', 'cuda']:
        cli = ROOT/('build-llama-' + backend)/'bin/llama-cli'
        if not cli.is_file():
            raise RuntimeError(f'Expected executable missing: {cli}; inspect pinned build outputs.')
        help_text = run_ice([cli, '--help'], cuda=backend == 'cuda', timeout=30)
        if '--no-conversation' not in help_text:
            raise RuntimeError('Pinned CLI differs: inspect --help and select its non-interactive completion mode before running.')
        run_ice([cli, '-m', MODEL_PATH, '-p', 'Say hello in one short sentence.', '-n', '16',
                 '-c', '2048', '-t', str(JOBS), '-ngl', '99' if backend == 'cuda' else '0',
                 '--temp', '0', '--no-conversation'], cuda=backend == 'cuda', timeout=180)
else:
    print('SKIP: complete optional baseline builds first.')

## 10. CPU and CUDA microbenchmarks + evidence manifest

One repetition is a smoke check, not a publishable comparison. The final system needs warmed-up, repeated end-to-end runs with client latency, scheduling and network costs. Review the pinned `llama-bench --help` if flag support differs.

In [ ]:
if BASELINE_BUILT:
    for backend in ['cpu', 'cuda']:
        bench = ROOT/('build-llama-' + backend)/'bin/llama-bench'
        run_ice([bench, '-m', MODEL_PATH, '-p', '32', '-n', '16', '-t', str(JOBS),
                 '-ngl', '99' if backend == 'cuda' else '0', '-r', '1'], cuda=backend == 'cuda', timeout=180)
    def revision(path):
        r = subprocess.run(['git', '-C', str(path), 'rev-parse', 'HEAD'], capture_output=True, text=True, timeout=15)
        return r.stdout.strip() if r.returncode == 0 else 'unavailable'
    manifest = dict(kind='llama.cpp-environment-smoke-NOT-Sutra', run_id=RUN_ID,
                    utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    host=socket.gethostname(), slurm_job_id=os.environ.get('SLURM_JOB_ID'),
                    repo_commit=revision(ROOT), llama_commit=revision(LLAMA),
                    model_path=str(MODEL_PATH), model_sha256=MODEL_SHA256,
                    model_source=MODEL_SOURCE, model_revision=MODEL_REVISION,
                    cuda_module=CUDA_MODULE, cc=SYSTEM_CC, cxx=SYSTEM_CXX,
                    cuda_arch=CUDA_ARCH, threads=JOBS, text_prompt='Say hello in one short sentence.',
                    max_new_tokens=16, context_tokens=2048, benchmark_prompt_tokens=32,
                    benchmark_generated_tokens=16, benchmark_repetitions=1)
    (RUN_DIR/'baseline_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Baseline evidence saved:', RUN_DIR)
else:
    print('SKIP: baseline disabled/not built.')

## 11. Sutra build and tests: implementation gate

This interface is **to implement**, described in the guide. The notebook never treats an empty CMake file or a test suite containing zero tests as success. Define `SUTRA_ENABLE_LLAMA`, actual build targets, and meaningful CTest cases before enabling `RUN_SUTRA`. The real CUDA mode also needs the staged model.

In [ ]:
SUTRA_BUILT = False
if IS_ICE_ALLOCATION and RUN_SUTRA:
    if not (ROOT/'CMakeLists.txt').read_text(encoding='utf-8').strip():
        raise RuntimeError('Sutra CMake is empty: implement M0/M1 first.')
    if DEMO_BACKEND not in {'mock', 'cuda'}:
        raise ValueError('DEMO_BACKEND must be mock or cuda.')
    if DEMO_BACKEND == 'cuda' and (not CUDA_PROBE_OK or MODEL_SHA256 is None):
        raise RuntimeError('CUDA mode requires a passing CUDA probe and a staged verified model.')
    build = ROOT/('build-' + DEMO_BACKEND)
    args = ['cmake', '-S', str(ROOT), '-B', str(build), '-DCMAKE_BUILD_TYPE=Release',
            '-DCMAKE_C_COMPILER=' + SYSTEM_CC, '-DCMAKE_CXX_COMPILER=' + SYSTEM_CXX,
            '-DSUTRA_ENABLE_LLAMA=' + ('ON' if DEMO_BACKEND == 'cuda' else 'OFF')]
    if DEMO_BACKEND == 'cuda':
        args += ['-DGGML_CUDA=ON', '-DCMAKE_CUDA_HOST_COMPILER=' + SYSTEM_CXX]
        if CUDA_ARCH:
            args += ['-DCMAKE_CUDA_ARCHITECTURES=' + str(CUDA_ARCH)]
    run_ice(args, cuda=DEMO_BACKEND == 'cuda', timeout=300)
    run_ice(['cmake', '--build', build, '--parallel', str(JOBS)], cuda=DEMO_BACKEND == 'cuda')
    run_ice(['ctest', '--test-dir', build, '--output-on-failure', '--no-tests=error'],
            cuda=DEMO_BACKEND == 'cuda', timeout=120)
    SUTRA_BUILT = True
else:
    print('PENDING/disabled: implement the C++ build/runtime, then enable RUN_SUTRA.')

## 12. Bounded end-to-end demonstration: implementation gate

Implement `scripts/run_demo.sh` first. It must start coordinator/worker processes, wait for readiness, run warmup then requests, save a unique run bundle, and clean up children with PID tracking/traps. CLI contract is in the guide. The following launches one worker; add verified two-GPU binding and failure scenarios in the launcher after that works. No service executable exists yet in the reviewed checkout.

In [ ]:
if SUTRA_BUILT:
    launcher = ROOT/'scripts/run_demo.sh'
    if not launcher.is_file():
        raise RuntimeError('Implement scripts/run_demo.sh with readiness, result saving and child cleanup first.')
    args = ['bash', launcher, '--backend', DEMO_BACKEND, '--workers', '1',
            '--requests', '100' if DEMO_BACKEND == 'mock' else '30',
            '--concurrency', '8' if DEMO_BACKEND == 'mock' else '4']
    if DEMO_BACKEND == 'cuda':
        args += ['--model', MODEL_PATH]
    run_ice(args, cuda=DEMO_BACKEND == 'cuda', timeout=300)
else:
    print('PENDING/disabled: no Sutra demo launched.')

## 13. Inspect saved real Sutra results

Set `SAVED_RESULTS_DIR` to the launcher's printed run directory. No sample/fake data is substituted. Expected CSV/manifest schema is in the guide. Numeric quantiles below use nearest-rank p95; a small smoke run cannot establish reliable tail latency. Throughput needs `measurement_wall_seconds` from the measured interval; model loading/warmup must be excluded by the runner.

In [ ]:
if SAVED_RESULTS_DIR is not None:
    folder = Path(SAVED_RESULTS_DIR).expanduser().resolve()
    with (folder/'requests.csv').open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    required = {'request_id', 'success', 'generated_tokens', 'client_latency_ms'}
    if not rows or not required.issubset(rows[0]):
        raise RuntimeError('Empty/incompatible requests.csv; use the agreed result contract.')
    if any(r['success'] not in {'true', 'false'} for r in rows):
        raise RuntimeError('CSV success values must be lowercase true/false.')
    ok = [r for r in rows if r['success'] == 'true']
    ids = [r['request_id'] for r in rows]
    print('Terminal rows:', len(rows), '| succeeded:', len(ok), '| failed:', len(rows)-len(ok))
    print('Unique request IDs:', len(set(ids)), '| duplicates:', len(ids)-len(set(ids)))
    if ok:
        latencies = sorted(float(r['client_latency_ms']) for r in ok)
        print('Client latency ms: median=', statistics.median(latencies),
              'p95(nearest rank)=', latencies[math.ceil(.95*len(latencies))-1])
    manifest = json.loads((folder/'manifest.json').read_text(encoding='utf-8'))
    seconds = manifest.get('measurement_wall_seconds')
    if isinstance(seconds, (int, float)) and seconds > 0:
        print('Successful requests/s:', len(ok)/seconds)
        print('Aggregate successful output tokens/s:', sum(int(r['generated_tokens']) for r in ok)/seconds)
    else:
        print('No positive measurement_wall_seconds: aggregate throughput unavailable.')
    for r in rows[:5]:
        print(r)
else:
    print('No result folder selected. Configure SAVED_RESULTS_DIR after a real Sutra run.')

## 14. Before the presentation

- Build Release and warm up before the demo; stage the model and check quota/time remaining.
- Show one real prompt, request ID, worker selection and generated text first.
- Compare FIFO/priority, actual multi-sequence batching, and one/two assigned GPUs on the same workload.
- Use the launcher to inject a worker failure and prove healthy-worker progress; never kill unrelated processes.
- Retain raw logs/CSV, configuration, revisions and measured wall time. Save the notebook with real outputs.
- Preserve a successful historical result bundle and label its run ID/hardware if live allocation is unavailable.
- End the interactive session when finished. Long measurements use separately tested Slurm jobs.

**Implementation order:** contract/build/mock -> FIFO scheduler + TCP path -> one real GPU worker -> priority/fairness/load balancing/faults -> true GPU batches -> experiments/report/demo. See the guide for owners, file boundaries, acceptance checks and dates through the tentative December 8 deadline.

Upstream references: [llama.cpp build](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md), [llama-bench](https://github.com/ggml-org/llama.cpp/blob/master/tools/llama-bench/README.md). Pin the repository revision; inspect its help/header if upstream APIs/flags have changed.